# MangaForge AI — Kaggle Dataset Pipeline
Bildsuche mit Wikimedia Commons, Lizenzprüfung und lokaler Trainingsvorbereitung. Metadaten sind keine Trainingserlaubnis.


In [1]:
from pathlib import Path
import csv, json, hashlib, shutil, urllib.request, urllib.parse, urllib.error, time, re
from PIL import Image, ImageStat
ROOT=Path('/kaggle/working/mangaforge') if Path('/kaggle').exists() else Path.cwd()/'mangaforge'
ROOT.mkdir(parents=True,exist_ok=True)
SOURCE_DIR=Path('/kaggle/input/mangaforge-images')
REVIEWED_MANIFEST=Path('/kaggle/input/mangaforge-reviewed-manifest/manifest.csv')
REVIEWED_LABELS=Path('/kaggle/input/mangaforge-reviewed-labels/training_review.csv')
EXTS={'.png','.jpg','.jpeg','.webp'}
def readcsv(p):
    with p.open(encoding='utf-8-sig',newline='') as f:return list(csv.DictReader(f))
def writecsv(p,rows,fields):
    with p.open('w',encoding='utf-8',newline='') as f:
        w=csv.DictWriter(f,fieldnames=fields,extrasaction='ignore');w.writeheader();w.writerows(rows)
def yes(x):return str(x).strip().lower() in ('yes','true','1','ja')
print('Working directory:',ROOT)


Working directory: /kaggle/working/mangaforge


## 1. Bildsuche: Wikimedia Commons mit kurzem Timeout
Openverse ist wegen wiederholter Timeouts standardmäßig deaktiviert. Ergebnisse sind nur **Prüfkandidaten**, keine Trainingsfreigabe.


In [2]:
# Standalone discovery: Wikimedia Commons first, Openverse optional.
from pathlib import Path
import csv, json, time, random, urllib.parse, urllib.request, urllib.error
ROOT=Path('/kaggle/working/mangaforge'); ROOT.mkdir(parents=True,exist_ok=True)
CANDIDATES=ROOT/'openverse_candidates.csv'  # legacy filename used by export cell
CHECKPOINT=ROOT/'discovery_completed.json'
FIELDS=['id','query','title','creator','license','license_url','source_url','image_url','width','height','attribution','commercial_allowed','ai_training_allowed','review_status']
SEARCHES=['manga illustration','anime illustration','comic illustration','ink drawing','character sketch','comic panel','samurai illustration','black and white illustration']
USE_OPENVERSE=False  # Enable only if its API becomes reachable
MAX_PAGES=2; PAGE_SIZE=20; TIMEOUT=8

def get_json(url):
    req=urllib.request.Request(url,headers={'User-Agent':'MangaForgeResearch/1.0 (dataset metadata review)','Accept':'application/json'})
    with urllib.request.urlopen(req,timeout=TIMEOUT) as resp:
        return json.load(resp)

def read_rows():
    if not CANDIDATES.exists(): return []
    with CANDIDATES.open(newline='',encoding='utf-8') as f: return list(csv.DictReader(f))

def save_rows(rows):
    with CANDIDATES.open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=FIELDS,extrasaction='ignore');w.writeheader();w.writerows(rows)

rows=read_rows(); seen={r['id'] for r in rows}; done=set(json.loads(CHECKPOINT.read_text()) if CHECKPOINT.exists() else [])
failures=0
for query in SEARCHES:
    for page in range(1,MAX_PAGES+1):
        key=f'commons::{query}::{page}'
        if key in done: continue
        params={'action':'query','generator':'search','gsrsearch':query,'gsrnamespace':6,'gsrlimit':PAGE_SIZE,'gsroffset':(page-1)*PAGE_SIZE,'prop':'imageinfo','iiprop':'url|size|extmetadata','format':'json','formatversion':'2'}
        url='https://commons.wikimedia.org/w/api.php?'+urllib.parse.urlencode(params)
        try:
            data=get_json(url)
            for item in data.get('query',{}).get('pages',[]):
                info=(item.get('imageinfo') or [{}])[0];meta=info.get('extmetadata',{})
                def mv(k): return meta.get(k,{}).get('value','')
                ident='commons:'+str(item.get('pageid',''))
                if ident in seen: continue
                rows.append({'id':ident,'query':query,'title':item.get('title',''),'creator':mv('Artist'),'license':mv('LicenseShortName'),'license_url':mv('LicenseUrl'),'source_url':info.get('descriptionurl',''),'image_url':info.get('url',''),'width':info.get('width',''),'height':info.get('height',''),'attribution':mv('Attribution'),'commercial_allowed':'unknown','ai_training_allowed':'unknown','review_status':'pending'})
                seen.add(ident)
            done.add(key); CHECKPOINT.write_text(json.dumps(sorted(done)));save_rows(rows)
            failures=0
            print('OK',query,page,'candidates:',len(rows),flush=True)
        except (TimeoutError, OSError, ValueError, urllib.error.URLError) as exc:
            failures+=1
            print('Source unavailable:',type(exc).__name__,str(exc)[:100],flush=True)
            if failures>=2:
                print('Stopping online discovery. Existing CSV is preserved.');break
        time.sleep(0.4)
    if failures>=2: break
if USE_OPENVERSE:
    print('Openverse option enabled; use a separate Openverse discovery cell after verifying connectivity.')
save_rows(rows)
print('Metadata candidates:',len(rows),'Saved:',CANDIDATES)
print('These are NOT approved for AI training; verify each source and license manually.')


OK manga illustration 1 candidates: 20
OK manga illustration 2 candidates: 40
OK anime illustration 1 candidates: 57
OK anime illustration 2 candidates: 76
OK comic illustration 1 candidates: 96
OK comic illustration 2 candidates: 116
OK ink drawing 1 candidates: 136
OK ink drawing 2 candidates: 156
OK character sketch 1 candidates: 176
OK character sketch 2 candidates: 196
OK comic panel 1 candidates: 216
OK comic panel 2 candidates: 235
OK samurai illustration 1 candidates: 235
OK samurai illustration 2 candidates: 255
OK black and white illustration 1 candidates: 275
OK black and white illustration 2 candidates: 295
Metadata candidates: 295 Saved: /kaggle/working/mangaforge/openverse_candidates.csv
These are NOT approved for AI training; verify each source and license manually.


## 2. Prepare local image inventory and verify rights

Upload images you own or have explicit permission to use into a Kaggle Dataset, attach it to the notebook, then point `SOURCE_DIR` to its folder under `/kaggle/input`. The inventory has `no` permissions by default. Copy the generated `manifest.csv` out, fill **source_url**, **license**, **commercial_allowed**, and **ai_training_allowed** based on evidence, and attach the reviewed manifest as a Kaggle Dataset for the next run. **Do not approve based on Openverse metadata alone.**


In [3]:
MANIFEST=ROOT/'manifest.csv'
fields=['path','sha256','license','source_url','permission_evidence','commercial_allowed','ai_training_allowed','review_status']
rows=[]
if SOURCE_DIR.is_dir():
    for p in sorted(SOURCE_DIR.rglob('*')):
        if not p.is_file() or p.suffix.lower() not in EXTS:continue
        try:
            with Image.open(p) as im: im.verify()
            digest=hashlib.sha256(p.read_bytes()).hexdigest()
            rows.append(dict(path=str(p),sha256=digest,license='',source_url='',permission_evidence='',commercial_allowed='no',ai_training_allowed='no',review_status='needs_manual_review'))
        except Exception as exc:print('Invalid image:',p.name,str(exc)[:90])
writecsv(MANIFEST,rows,fields)
print('Inventory:',len(rows),'Review CSV:',MANIFEST)


Inventory: 0 Review CSV: /kaggle/working/mangaforge/manifest.csv


In [4]:
APPROVED=ROOT/'approved_dataset';APPROVED.mkdir(exist_ok=True)
accepted=[]; rejected=[];seen=set()
if REVIEWED_MANIFEST.is_file():
    for row in readcsv(REVIEWED_MANIFEST):
        raw=row.get('path','');p=Path(raw)
        reason=''
        if not raw or not p.is_file() or p.suffix.lower() not in EXTS:reason='missing_or_invalid_path'
        elif not yes(row.get('commercial_allowed')) or not yes(row.get('ai_training_allowed')):reason='rights_not_approved'
        elif row.get('review_status','').strip().lower() not in ('approved','verified'):reason='review_not_approved'
        elif not row.get('permission_evidence','').strip():reason='missing_permission_evidence'
        else:
            digest=hashlib.sha256(p.read_bytes()).hexdigest()
            if row.get('sha256','').strip().lower()!=digest:reason='hash_mismatch'
            elif digest in seen:reason='duplicate'
            else:
                try:
                    with Image.open(p) as im:im.verify()
                    dst=APPROVED/(digest[:20]+p.suffix.lower());shutil.copy2(p,dst)
                    accepted.append({**row,'imported_path':str(dst)});seen.add(digest)
                except Exception:reason='invalid_image'
        if reason:rejected.append({**row,'reason':reason})
writecsv(APPROVED/'accepted.csv',accepted,fields+['imported_path'])
writecsv(APPROVED/'rejected.csv',rejected,fields+['reason'])
print('Accepted:',len(accepted),'Rejected:',len(rejected))


Accepted: 0 Rejected: 0


## 3. Quality audit and provisional classification

Technical audit uses dimensions, contrast and perceptual hashing. The provisional category below is inferred from the *filename*, not from an AI vision model. Manual visual review remains necessary.


In [5]:
quality=[]; review=[]
def category_for(name):
    name=name.lower()
    for word,cat in [('panel','comic_panel'),('face','portrait'),('character','character'),('landscape','background'),('action','action')]:
        if word in name:return cat
    return 'unclassified'
for row in accepted:
    p=Path(row['imported_path'])
    with Image.open(p) as im:
        w,h=im.size; gray=im.convert('L'); contrast=ImageStat.Stat(gray).stddev[0]
        tiny=gray.resize((8,8)); vals=list(tiny.getdata());avg=sum(vals)/len(vals)
        phash=''.join('1' if x>avg else '0' for x in vals)
    flag='review' if min(w,h)<512 or contrast<15 else 'pass'
    quality.append({'imported_path':str(p),'width':w,'height':h,'contrast':round(contrast,2),'a_hash':phash,'quality_status':flag})
    review.append({'imported_path':str(p),'category_suggestion':category_for(p.name),'category_verified':'','caption_verified':'','training_approved':'no'})
writecsv(APPROVED/'quality_report.csv',quality,['imported_path','width','height','contrast','a_hash','quality_status'])
writecsv(ROOT/'training_review.csv',review,['imported_path','category_suggestion','category_verified','caption_verified','training_approved'])
print('Quality audit:',len(quality),'Review labels:',ROOT/'training_review.csv')


Quality audit: 0 Review labels: /kaggle/working/mangaforge/training_review.csv


## 4. Training preparation (not training)

Only entries independently marked `training_approved=yes`, with a verified category and caption, enter `train.jsonl`. The notebook does not train a model or claim the output is legally cleared by itself. Use only a trusted, reviewed `training_review.csv`.


In [6]:
TRAIN=ROOT/'train.jsonl';count=0
approved_paths={str(Path(r['imported_path']).resolve()) for r in accepted}
quality_ok={r['imported_path'] for r in quality if r['quality_status']=='pass'}
with TRAIN.open('w',encoding='utf-8') as out:
    if REVIEWED_LABELS.is_file():
        for row in readcsv(REVIEWED_LABELS):
            raw=row.get('imported_path','');p=Path(raw)
            if not raw or str(p.resolve()) not in approved_paths or str(p) not in quality_ok or not p.is_file():continue
            if not yes(row.get('training_approved')):continue
            category=row.get('category_verified','').strip();caption=row.get('caption_verified','').strip()
            if not category or not caption:continue
            out.write(json.dumps({'image':str(p),'caption':caption,'category':category},ensure_ascii=False)+'\n');count+=1
print('Training records:',count,'JSONL:',TRAIN)


Training records: 0 JSONL: /kaggle/working/mangaforge/train.jsonl


### Notes
- Kaggle `/kaggle/input` is read-only. Output paths in `/kaggle/working` are temporary unless saved as notebook output/version.
- Image licenses, model training permissions, and publication rights are separate questions; consult the original rights holder and terms.
- Large-scale dataset work should be incremental, within Kaggle and source API limits.


## Export and GitHub integration
Download notebook and output files from Kaggle, then commit them to your repository. No GitHub credentials are required inside Kaggle. **Never commit private training images, tokens or permission documents.** GitHub integration is not performed automatically.


In [7]:
import zipfile
bundle=ROOT/'mangaforge_reports.zip'
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
    for f in [ROOT/'openverse_candidates.csv',ROOT/'manifest.csv',ROOT/'training_review.csv',ROOT/'train.jsonl',APPROVED/'accepted.csv',APPROVED/'rejected.csv',APPROVED/'quality_report.csv']:
        if f.is_file():z.write(f,f.relative_to(ROOT))
print('Download reports:',bundle)


Download reports: /kaggle/working/mangaforge/mangaforge_reports.zip


In [8]:
import pandas as pd

path = "/kaggle/working/mangaforge/openverse_candidates.csv"
df = pd.read_csv(path)

print("Gesamt:", len(df))
print("\nSpalten:", df.columns.tolist())
print("\nLizenzen:")
print(df["license"].value_counts(dropna=False) if "license" in df else "Keine Lizenzspalte")
print("\nDuplikate:")
print(df.duplicated().sum())


Gesamt: 295

Spalten: ['id', 'query', 'title', 'creator', 'license', 'license_url', 'source_url', 'image_url', 'width', 'height', 'attribution', 'commercial_allowed', 'ai_training_allowed', 'review_status']

Lizenzen:
license
Public domain           140
No restrictions          35
CC BY 2.0                32
CC0                      27
CC BY-SA 3.0             23
CC BY-SA 4.0             18
CC BY-SA 2.0             13
CC BY 3.0                 4
CC BY 4.0                 2
Copyrighted free use      1
Name: count, dtype: int64

Duplikate:
0


In [9]:
from pathlib import Path

print("Suche MangaForge-Dateien ...")

for base in ["/kaggle/working", "/kaggle/input"]:
    for p in Path(base).rglob("*"):
        if p.is_file() and (
            "candidates" in p.name.lower()
            or "mangaforge" in p.name.lower()
        ):
            print(p)
            

Suche MangaForge-Dateien ...
/kaggle/working/mangaforge/openverse_candidates.csv
/kaggle/working/mangaforge/mangaforge_reports.zip


In [10]:
from pathlib import Path
from IPython.display import FileLink

csv_file = Path(
    "/kaggle/working/mangaforge/openverse_candidates.csv"
)

if csv_file.exists():
    print("CSV gefunden!")
    print("Dateigröße:", csv_file.stat().st_size, "Bytes")
    display(FileLink(str(csv_file)))
else:
    print("CSV fehlt – Bildsuche zuerst starten.")
    

CSV gefunden!
Dateigröße: 163214 Bytes


/kaggle/working/mangaforge/openverse_candidates.csv